|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Guided decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: make invalid output unreachable<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np

from tests.helpers import json_prefix_state

rng = np.random.default_rng(0)
# a vocabulary of multi-character pieces, like a real tokenizer
VOCAB = ['{', '}', '[', ']', '"', ':', ',', ' ', '1', '2', '42',
         'true', 'false', 'null', 'name', 'age', '": ', '", "', ': {', '}, ']
print(f'{len(VOCAB)} tokens, and most of them are more than one character')

20 tokens, and most of them are more than one character


Constrain the output, so that invalid JSON is unreachable and not only
improbable.

You get the grammar. `json_prefix_state` classifies a string as `valid`,
`prefix` or `invalid`. Your work is the part that touches the sampler, and the
part that makes the mask affordable.

# Exercise 1: which tokens are legal here?

Note that the vocabulary contains multi-character pieces like `'": '`. That
is what makes real guided decoding fiddly: the FSM advances by a whole token,
not a character.

In [2]:
def legal(prefix):
  """Which tokens can follow `prefix` without making it unrecoverable?"""
  return np.array([json_prefix_state(prefix + t) != 'invalid' for t in VOCAB])

for p in ['', '{', '{"', '{"name', '{"name"', '{"name": ']:
  m = legal(p)
  print(f'{p!r:<12} {m.sum():>2}/{len(VOCAB)}: {[t for t,k in zip(VOCAB,m) if k][:8]}')

''           12/20: ['{', '[', '"', ' ', '1', '2', '42', 'true']
'{'           5/20: ['}', '"', ' ', '": ', '", "']
'{"'         19/20: ['{', '}', '[', ']', '"', ':', ',', ' ']
'{"name'     19/20: ['{', '}', '[', ']', '"', ':', ',', ' ']
'{"name"'     3/20: [':', ' ', ': {']
'{"name": '  12/20: ['{', '[', '"', ' ', '1', '2', '42', 'true']


# Exercise 2: sample with the mask on

Two hundred runs with random logits. Not one of them should produce invalid
JSON.

In [3]:
def masked_sample(prefix, logits):
  m = legal(prefix)
  if not m.any(): return None
  masked = np.where(m, logits, -np.inf)
  p = np.exp(masked - masked.max()); p = p/p.sum()
  return VOCAB[int(rng.choice(len(VOCAB), p=p))]

bad = 0
for trial in range(200):
  prefix = ''
  for _ in range(14):
    t = masked_sample(prefix, rng.normal(size=len(VOCAB)))
    if t is None: break
    prefix += t
    if json_prefix_state(prefix) == 'valid' and len(prefix) > 6: break
  if json_prefix_state(prefix) == 'invalid': bad += 1
print(f'{bad}/200 samples produced invalid JSON')
print(f'example: {prefix!r}  ({json_prefix_state(prefix)})')

0/200 samples produced invalid JSON
example: '[null,1,": [:name[,[42null'  (prefix)


# Exercise 3: where the one-step mask is not enough

Count the runs that reach a prefix with no legal continuation at all.

In [4]:
states = {'valid':0, 'prefix':0, 'invalid':0}
examples = []
for trial in range(500):
  prefix = ''
  for step in range(14):                 # a length limit, as a server has
    t = masked_sample(prefix, rng.normal(size=len(VOCAB)))
    if t is None: break
    prefix += t
  s = json_prefix_state(prefix)
  states[s] += 1
  if s == 'prefix' and len(examples) < 3: examples.append(prefix)

for k, v in states.items():
  print(f'{k:>8}: {v:>4}/500')
print('\nincomplete examples, stopped by the length limit:')
for e in examples: print(f'  {e!r}')

   valid:  402/500
  prefix:   98/500
 invalid:    0/500

incomplete examples, stopped by the length limit:
  '":agenull 242true: {}, ]age: {:'
  '{ "2false:": {"}, ,[ ,'
  '{", ":  true,  "2422'


### Two results, and people ship without the second one

**Zero invalid outputs.** That is the guarantee, and it is real. A prompt that
asks politely for JSON gets JSON most of the time. A mask gets JSON every
time.

**And one fifth of the runs never finish.** Look at the `prefix` count. The
mask promises that the string can still become valid. It does not promise that
the string becomes valid. A run that reaches its length limit with open
brackets gives the user a truncated object, and that object parses as nothing.

So "valid JSON, guaranteed" is two guarantees, and the mask gives you one of
them. For the other one, the FSM must know how many brackets are open. It can
then force the closing brackets as the limit comes near.

This is why a real implementation compiles the grammar to an automaton, and
does not validate a string again at each step. The validator here is a
teaching aid.

### And the cost that you have not paid yet

`legal()` calls the validator one time for each token in the vocabulary. With
20 tokens that is free. With 151,936 tokens it costs approximately thirty
decode steps.

The mask depends only on the FSM **state**, and the number of states is
finite. Cache one mask for each state. Build the next mask on another thread
while the GPU is busy.

Stage 19 measures the milliseconds. A correctness feature that halves your
throughput is a feature that somebody turns off.

    ./vc guide 19